# Chapter 1 Quiz Solutions & Python Validations

This Jupyter Notebook contains comprehensive answers, theoretical proofs, and numerical Python simulations for all 5 questions from the **Section 1.7 Chapter Quiz** in *Causal Inference for Data Science* (2025).

---

## Question 1
**What is the difference between observational and experimental data?**

### Detailed Answer:
- **Experimental Data (RCTs / A/B Tests)**: Generated under controlled conditions where the researcher directly manipulates the treatment assignment ($X$) using a random mechanism (e.g., coin toss). Random assignment ensures that treatment $X$ is statistically independent of all baseline covariates and potential confounders $Z$ ($X \perp Z$).
- **Observational Data**: Collected without active researcher intervention. Treatment choice is determined by nature, individual choice, or business logic, which often depends on confounding variables $Z$. As a result, $X 
ot\perp Z$, leading to confounding bias where correlation does not equal causation ($P(Y|X) 
eq P(Y|do(X))$).


In [ ]:
# Code Verification for Question 1: Demonstrating Independence in RCT vs Correlation in Observational Data

np.random.seed(42)
n = 10000

# Confounder: Customer Purchasing Power / Income
income = np.random.normal(loc=50, scale=10, size=n)

# Observational Assignment: High-income customers opt-in to Premium Tier (Treatment)
prob_treatment_obs = 1 / (1 + np.exp(-(income - 50) / 5))
treatment_obs = np.random.binomial(1, prob_treatment_obs)

# RCT Assignment: Random coin flip completely independent of Income
treatment_rct = np.random.binomial(1, 0.5, size=n)

# Compute correlations with confounder (Income)
corr_obs = np.corrcoef(treatment_obs, income)[0, 1]
corr_rct = np.corrcoef(treatment_rct, income)[0, 1]

print(f"Correlation between Confounder (Income) & Observational Treatment: {corr_obs:.4f}  (Strong Confounding)")
print(f"Correlation between Confounder (Income) & RCT Treatment:           {corr_rct:.4f}  (No Confounding)")



---
## Question 2
**When should you run an A/B test or RCT?**

### Detailed Answer:
According to **Section 1.3.5**, you should **ALWAYS** run an A/B test or RCT whenever it is feasible, ethical, cost-effective, and timely, because RCTs represent the gold standard for establishing causal relationships.

Conversely, you must rely on observational causal inference methods when:
1. **Experimentation is Infeasible**: You cannot force competitors to randomize product releases or pricing.
2. **Experimentation is Unethical**: You cannot randomly assign participants to harmful behaviors (e.g., smoking during pregnancy).
3. **Experimentation is Costly or Timely**: Longitudinal studies observing multi-year health outcomes take decades.
4. **Lack of External Validity**: Volunteer trial participants may not reflect the general population.


---
## Question 3
**When running an A/B test, if your sample is large enough, both groups will have the same population characteristics. Why?**

### Detailed Answer:
According to **Section 1.3.4 (Step 3: Analysis)**, when treatment is assigned completely at random (e.g., coin toss per user ID), every participant has an equal probability of assignment regardless of their baseline traits (age, income, location, pre-existing health status).

By the **Law of Large Numbers (LLN)** and the **Glivenko-Cantelli Theorem**, as the sample size $n ightarrow \infty$, the empirical distribution of covariates in Group A ($	ilde{F}_A(Z)$) and Group B ($	ilde{F}_B(Z)$) both converge uniformly to the true population distribution $F(Z)$. Consequently, baseline characteristics are balanced between groups.


In [ ]:
# Code Verification for Question 3: Demonstrating Covariate Balance via Randomization

def check_covariate_balance(sample_size):
    # Simulated population traits: Age ~ Normal(40, 10), Income ~ Normal(60k, 15k)
    age = np.random.normal(40, 10, size=sample_size)
    income = np.random.normal(60000, 15000, size=sample_size)
    
    # Random assignment (50/50 split)
    group_assignment = np.random.binomial(1, 0.5, size=sample_size)
    
    group_A_age, group_B_age = age[group_assignment == 1].mean(), age[group_assignment == 0].mean()
    group_A_inc, group_B_inc = income[group_assignment == 1].mean(), income[group_assignment == 0].mean()
    
    return abs(group_A_age - group_B_age), abs(group_A_inc - group_B_inc)

print("=== COVARIATE BALANCE VS SAMPLE SIZE ===")
for size in [50, 500, 5000, 500000]:
    age_diff, inc_diff = check_covariate_balance(size)
    print(f"Sample Size: {size:6d} | Age Mean Diff: {age_diff:6.3f} yrs | Income Mean Diff: ${inc_diff:7.2f}")



---
## Question 4
**Give one reason correlations do not always provide the right evidence for causation.**

### Detailed Answer:
According to **Section 1.4**, a primary reason correlation does not equal causation is the presence of **Confounding Variables (Common Causes)**. A third variable $Z$ can simultaneously influence $X$ and $Y$, generating a strong statistical correlation ($corr(X,Y) 
eq 0$) even when no direct causal mechanism exists between $X$ and $Y$ ($X 
otightarrow Y$).

Additionally, correlation is **symmetric** ($corr(X,Y) = corr(Y,X)$), whereas causation is **directional and asymmetric** ($X ightarrow Y 
eq Y ightarrow X$).


In [ ]:
# Code Verification for Question 4: Spurious Correlation via Common Cause Z

n = 5000
# Z = Temperature / Season
Z = np.random.uniform(10, 35, size=n)

# X = Ice Cream Sales (driven by Temperature Z)
X = 50 + 3.5 * Z + np.random.normal(0, 5, size=n)

# Y = Drowning Accidents (driven by Temperature Z, people swim when hot)
Y = 5 + 0.8 * Z + np.random.normal(0, 2, size=n)

# Correlation between X and Y
correlation = np.corrcoef(X, Y)[0, 1]
print(f"Correlation between Ice Cream Sales (X) & Drowning Accidents (Y): {correlation:.4f}")
print("Conclusion: Selling less ice cream will NOT reduce drowning accidents! Both are driven by Temperature (Z).")



---
## Question 5
**Which confounders can you find in an A/B test?**

### Detailed Answer:
According to **Section 1.4.2**, in a properly executed A/B test or RCT, there are **NONE (zero confounders)**.

### Theoretical Proof:
By definition, a confounder $Z$ is a variable that affects **both** the treatment decision $X$ and the outcome $Y$ ($X \leftarrow Z ightarrow Y$). In a randomized experiment, treatment $X$ is determined solely by an external random number generator ($Random ightarrow X$). Because baseline covariates $Z$ have zero influence over $X$, the link $Z ightarrow X$ is severed. Without dual influence, baseline variables cannot act as confounders.
